# INF 369 Final Project — Almaty Housing Market (Krisha.kz)

**Полный pipeline в одном ноутбуке:**

1. **Task 1** — Scraper (`scraper.py`) → `krisha_raw.csv`
2. **Task 2** — Preprocessing (`preprocessing.py`) → `krisha_cleaned.csv`
3. **Task 3** — Analysis (`analysis.py`) → 5 аналитических вопросов
4. **Task 4** — Visualization (`visualization.py`) → 10 графиков в `charts/`
5. **Task 5** — Summary report (`main.py`) → ключевые insights
6. **Run full pipeline** — запускает всё end-to-end

Запускайте ячейки по порядку, либо перейдите сразу к секции **«Run full pipeline»** в самом низу.

## 0. Imports & global config

In [ ]:
import sys
import time
import random
import re
import logging
import warnings
from pathlib import Path
from datetime import date
from concurrent.futures import ThreadPoolExecutor, as_completed
from typing import Optional, List, Dict, Any, Set, Tuple

import requests
from bs4 import BeautifulSoup
import pandas as pd
import numpy as np

import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import plotly.express as px

warnings.filterwarnings("ignore")
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[
        logging.StreamHandler(sys.stdout),
        logging.FileHandler("pipeline.log", encoding="utf-8"),
    ],
)
logger = logging.getLogger(__name__)

RAW_CSV = "krisha_raw.csv"
CLEAN_CSV = "krisha_cleaned.csv"
SCRAPE_PAGES = None
SCRAPE_MAX_PAGES_CAP = 10_000
BUILD_YEAR_SPAN = 2
SCRAPE_MAX_LISTINGS = 2000  # Cap on Pass B (None = full crawl)

print("Working directory:", Path.cwd())

## 1. Task 1 — Scraper (krisha.kz)

Two-pass scraper с фильтром Krisha по году постройки (`das[house.year]`):

- **Pass A** — `das[novostroiki]=1` + year window: собираем official ID новостроек.
- **Pass B** — полный каталог (с тем же year-окном); `Is_NewBuild = True` только если ID попал в множество из Pass A.
- **Pass C** *(опционально)* — детальные страницы листингов параллельно: добивает `Developer` / `Build_Year` / `House_Type` / `Ceiling_Height` / `Bathroom`.

In [ ]:
BASE_URL = "https://krisha.kz/prodazha/kvartiry/almaty/"
NEWBUILDINGS_QUERY: Dict[str, Any] = {"das[novostroiki]": "1"}
DEFAULT_BUILD_YEAR_SPAN = 2
DEFAULT_MAX_PAGES_CAP = 10_000
DEFAULT_DETAIL_WORKERS = 8
DEFAULT_DETAIL_DELAY_RANGE = (0.15, 0.45)
DEFAULT_DETAIL_TIMEOUT: tuple = (5, 10)
DEFAULT_DETAIL_LOG_EVERY = 25
DEFAULT_DETAIL_CHECKPOINT_EVERY = 100
DETAIL_DONE_COLUMN = "Detail_Fetched"
MIN_VALID_BUILD_YEAR = 1900
MAX_VALID_BUILD_YEAR = date.today().year + 5

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/122.0.0.0 Safari/537.36"
    ),
    "Accept-Language": "ru-RU,ru;q=0.9,en;q=0.8",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Referer": "https://krisha.kz/",
}

_HOUSE_TYPE_KEYWORDS = (
    "монолитный", "кирпичный", "панельный", "блочный",
    "каркасно-камышитовый", "каркасный", "иное",
)
_BATHROOM_KEYWORDS = (
    "2 с/у и более", "2 санузла", "санузел совмещенный", "санузел раздельный",
    "совмещенный", "раздельный",
)
_DETAIL_LABEL_MAP = {
    "тип дома": "House_Type",
    "жилой комплекс": "Residential_Complex",
    "год постройки": "Build_Year",
    "санузел": "Bathroom",
    "высота потолков": "Ceiling_Height",
    "застройщик": "Developer",
}

In [ ]:
def _parse_price(text: str) -> Optional[int]:
    cleaned = re.sub(r"[^\d]", "", text.strip())
    return int(cleaned) if cleaned else None


def _parse_title(title: str) -> dict:
    result = {"Title": title.strip(), "Rooms": None, "Area_m2": None, "Floor_Info": None}
    rooms_match = re.search(r"(\d+)-комнатная", title)
    if rooms_match:
        result["Rooms"] = int(rooms_match.group(1))
    elif "студия" in title.lower():
        result["Rooms"] = 0
    area_match = re.search(r"([\d.,]+)\s*м²", title)
    if area_match:
        result["Area_m2"] = float(area_match.group(1).replace(",", "."))
    floor_match = re.search(r"(\d+)/(\d+)\s*этаж", title)
    if floor_match:
        result["Floor_Info"] = f"{floor_match.group(1)} из {floor_match.group(2)}"
    else:
        single_floor = re.search(r"(\d+)\s*этаж", title)
        if single_floor:
            result["Floor_Info"] = single_floor.group(1)
    return result


def _parse_location(subtitle: str) -> dict:
    result = {"District": None, "Address": None}
    if not subtitle:
        return result
    parts = subtitle.split(",", 1)
    result["District"] = parts[0].strip()
    result["Address"] = parts[1].strip() if len(parts) > 1 else None
    return result


def _year_window_params(year_span: int = DEFAULT_BUILD_YEAR_SPAN, year_end: Optional[int] = None) -> Dict[str, str]:
    if year_span < 1:
        raise ValueError("year_span must be >= 1")
    end = year_end if year_end is not None else date.today().year
    y_from = end - (year_span - 1)
    return {"das[house.year][from]": str(y_from), "das[house.year][to]": str(end)}


def _build_params(page: int, newbuildings_only: bool, year_params: Dict[str, str]) -> Dict[str, Any]:
    params: Dict[str, Any] = dict(year_params)
    if newbuildings_only:
        params.update(NEWBUILDINGS_QUERY)
    if page > 1:
        params["page"] = page
    return params


def _fetch_soup(session: requests.Session, page: int, *, newbuildings_only: bool, year_params: Dict[str, str]) -> Optional[BeautifulSoup]:
    params = _build_params(page, newbuildings_only, year_params)
    try:
        resp = session.get(BASE_URL, params=params, headers=HEADERS, timeout=20)
        resp.raise_for_status()
    except requests.RequestException as exc:
        logger.warning("Fetch page %d (newbuild=%s) failed: %s", page, newbuildings_only, exc)
        return None
    return BeautifulSoup(resp.text, "lxml")


def _listing_ids_from_soup(soup: BeautifulSoup) -> List[str]:
    cards = soup.find_all("div", class_="a-card")
    return [str(c.get("data-id")) for c in cards if c.get("data-id")]

In [ ]:
def infer_newbuild_from_text(description: str, title: str) -> bool:
    return _infer_newbuild_heuristic(description or "", title or "")


def _infer_newbuild_heuristic(description: str, title: str) -> bool:
    desc = (description or "").lower()
    blob = f"{desc} {(title or '').lower()}"
    if re.search(r"панельный дом,\s*19[0-8]\d", desc):
        return False
    if re.search(r"кирпичный дом,\s*19[0-6]\d", desc):
        return False
    if "от застройщика" in blob:
        return True
    if re.search(r"новостро", blob):
        return True
    if "жил. комплекс" in desc or re.search(r"\bжк\s", desc) or "жк «" in desc:
        years = [int(y) for y in re.findall(r"(20\d{2})\s*г\.п", desc)]
        if years and max(years) >= 2015:
            return True
    years = [int(y) for y in re.findall(r"(20[2-9]\d|201[6-9])\s*г\.п", desc)]
    if years and max(years) >= 2016:
        if "панельный дом, 19" not in desc:
            return True
    return False


def _classify_newbuild(listing_id: str, official_ids: Set[str]) -> bool:
    if not listing_id:
        return False
    return str(listing_id) in official_ids


def _to_float_meters(text: str) -> Optional[float]:
    if not text:
        return None
    m = re.search(r"([\d]+[.,]?\d*)", text.replace(",", "."))
    if not m:
        return None
    try:
        val = float(m.group(1))
    except ValueError:
        return None
    if 1.5 <= val <= 6.0:
        return round(val, 2)
    return None


def _valid_build_year(year: Optional[int]) -> Optional[int]:
    if year is None:
        return None
    if MIN_VALID_BUILD_YEAR <= year <= MAX_VALID_BUILD_YEAR:
        return year
    return None

In [ ]:
def _parse_snippet_details(description: str) -> Dict[str, Any]:
    out: Dict[str, Any] = {
        "Build_Year": None,
        "Residential_Complex": None,
        "House_Type": None,
        "Ceiling_Height": None,
        "Bathroom": None,
    }
    if not description:
        return out
    desc = description.strip()
    desc_lc = desc.lower()

    yr = re.search(r"(19\d{2}|20\d{2})\s*г\.?п", desc_lc)
    if yr:
        out["Build_Year"] = _valid_build_year(int(yr.group(1)))

    rc = re.search(r"жил\.\s*комплекс\s+([^,]+?)(?=,)", desc, flags=re.IGNORECASE)
    if not rc:
        rc = re.search(r"ЖК\s+«([^»]+)»", desc)
    if not rc:
        rc = re.search(r"\bЖК\s+([^,]+?)(?=,)", desc)
    if rc:
        name = rc.group(1).strip().strip("«»\"' ")
        if name:
            out["Residential_Complex"] = name

    ht_match = re.search(r"\b(монолитн\w+|кирпичн\w+|панельн\w+|блочн\w+|каркасн\w+)\s+дом", desc_lc)
    if ht_match:
        token = ht_match.group(1)
        for canon in _HOUSE_TYPE_KEYWORDS:
            if canon.startswith(token[:6]):
                out["House_Type"] = canon
                break

    ch = re.search(r"потолк\w*\s*([\d.,]+)\s*м", desc_lc)
    if ch:
        out["Ceiling_Height"] = _to_float_meters(ch.group(1))

    if re.search(r"\b2\s*с/у\b|\b2\s*санузла", desc_lc):
        out["Bathroom"] = "2 санузла и более"
    else:
        b = re.search(r"санузел\s+(совмещ\w+|раздельн\w+)", desc_lc)
        if b:
            out["Bathroom"] = "совмещенный" if b.group(1).startswith("совмещ") else "раздельный"
    return out


def _coerce_detail_value(field: str, raw: str) -> Any:
    text = raw.strip()
    if not text:
        return None
    if field == "Build_Year":
        m = re.search(r"(19\d{2}|20\d{2})", text)
        return _valid_build_year(int(m.group(1))) if m else None
    if field == "Ceiling_Height":
        return _to_float_meters(text)
    if field == "House_Type":
        low = text.lower()
        for canon in _HOUSE_TYPE_KEYWORDS:
            if canon in low:
                return canon
        return text
    if field == "Bathroom":
        low = text.lower()
        if "2" in low and ("с/у" in low or "санузл" in low):
            return "2 санузла и более"
        if "совмещ" in low:
            return "совмещенный"
        if "раздельн" in low:
            return "раздельный"
        return text
    return text


def _parse_detail_page(soup: BeautifulSoup) -> Dict[str, Any]:
    facts: Dict[str, Any] = {k: None for k in _DETAIL_LABEL_MAP.values()}

    items = soup.find_all("div", class_="offer__info-item")
    for item in items:
        title_el = item.find(class_="offer__info-title")
        value_el = item.find(class_="offer__info-value")
        if not title_el or not value_el:
            continue
        label = title_el.get_text(strip=True).lower()
        field = _DETAIL_LABEL_MAP.get(label)
        if not field:
            continue
        value = _coerce_detail_value(field, value_el.get_text(" ", strip=True))
        if value is not None and facts.get(field) in (None, ""):
            facts[field] = value

    if not any(facts.values()):
        for dl in soup.find_all("dl", class_="offer__parameters"):
            for dt, dd in zip(dl.find_all("dt"), dl.find_all("dd")):
                label = dt.get_text(strip=True).lower()
                field = _DETAIL_LABEL_MAP.get(label)
                if not field:
                    continue
                value = _coerce_detail_value(field, dd.get_text(" ", strip=True))
                if value is not None and facts.get(field) in (None, ""):
                    facts[field] = value

    if not facts.get("Developer"):
        for owner_el in soup.find_all(class_=re.compile(r"owners__name|builder__name|developer__title")):
            text = owner_el.get_text(" ", strip=True)
            if not text:
                continue
            stripped = re.sub(r"^застройщик\s+", "", text, flags=re.IGNORECASE).strip()
            if stripped and stripped.lower() != text.lower():
                facts["Developer"] = stripped
                break
    return facts


def _fetch_listing_details(session: requests.Session, url: str, *, timeout: tuple = DEFAULT_DETAIL_TIMEOUT) -> Dict[str, Any]:
    try:
        resp = session.get(url, headers=HEADERS, timeout=timeout)
        resp.raise_for_status()
    except requests.RequestException as exc:
        logger.debug("Detail fetch failed for %s: %s", url, exc)
        return {}
    soup = BeautifulSoup(resp.text, "lxml")
    return _parse_detail_page(soup)


def _enrich_with_details(
    records: List[dict],
    *,
    workers: int = DEFAULT_DETAIL_WORKERS,
    delay_range: tuple = DEFAULT_DETAIL_DELAY_RANGE,
    request_timeout: tuple = DEFAULT_DETAIL_TIMEOUT,
    checkpoint_path: Optional[str] = None,
    checkpoint_every: int = DEFAULT_DETAIL_CHECKPOINT_EVERY,
    log_every: int = DEFAULT_DETAIL_LOG_EVERY,
) -> None:
    """Resumable + checkpointed detail enrichment.

    Skips records where ``Detail_Fetched`` is already True and dumps the full list to
    ``checkpoint_path`` every ``checkpoint_every`` completions, so Ctrl+C never wipes
    progress.
    """
    if not records:
        return

    pending = [r for r in records if not r.get(DETAIL_DONE_COLUMN)]
    already = len(records) - len(pending)
    if not pending:
        logger.info("All %d records already have %s=True — nothing to enrich.", len(records), DETAIL_DONE_COLUMN)
        return

    logger.info(
        "Pass C: enriching %d/%d records (already done %d, workers=%d, timeout=%s, checkpoint every %d, log every %d)…",
        len(pending), len(records), already, workers, request_timeout, checkpoint_every, log_every,
    )

    session = requests.Session()
    completed = 0
    start = time.monotonic()

    def _save_checkpoint(tag: str) -> None:
        if not checkpoint_path:
            return
        pd.DataFrame(records).to_csv(checkpoint_path, index=False, encoding="utf-8-sig")
        logger.info("  checkpoint (%s) → %s", tag, checkpoint_path)

    def worker(rec: dict) -> tuple:
        time.sleep(random.uniform(*delay_range))
        url = rec.get("Listing_URL")
        if not url:
            return rec, {}
        return rec, _fetch_listing_details(session, url, timeout=request_timeout)

    try:
        with ThreadPoolExecutor(max_workers=workers) as pool:
            futures = [pool.submit(worker, rec) for rec in pending]
            for fut in as_completed(futures):
                try:
                    rec, detail = fut.result()
                except Exception as exc:  # noqa: BLE001
                    logger.debug("Detail worker error: %s", exc)
                    completed += 1
                    continue
                for field, value in detail.items():
                    if value is None:
                        continue
                    if rec.get(field) in (None, ""):
                        rec[field] = value
                rec[DETAIL_DONE_COLUMN] = True
                completed += 1

                if completed % log_every == 0 or completed == len(pending):
                    elapsed = max(time.monotonic() - start, 1e-6)
                    rate = completed / elapsed
                    remaining = len(pending) - completed
                    eta_min = remaining / rate / 60 if rate > 0 else float("inf")
                    logger.info("  details %d/%d (%.2f/s, ETA %.1f min)", completed, len(pending), rate, eta_min)

                if checkpoint_path and checkpoint_every > 0 and completed % checkpoint_every == 0:
                    _save_checkpoint(f"{completed}/{len(pending)}")
    finally:
        _save_checkpoint("final")


def enrich_csv_with_details(
    csv_path: str = "krisha_raw.csv",
    *,
    workers: int = DEFAULT_DETAIL_WORKERS,
    request_timeout: tuple = DEFAULT_DETAIL_TIMEOUT,
    checkpoint_every: int = DEFAULT_DETAIL_CHECKPOINT_EVERY,
    log_every: int = DEFAULT_DETAIL_LOG_EVERY,
) -> pd.DataFrame:
    """Resume Pass C on an existing raw CSV without redoing Pass A+B."""
    df = pd.read_csv(csv_path, encoding="utf-8-sig")
    if DETAIL_DONE_COLUMN not in df.columns:
        df[DETAIL_DONE_COLUMN] = False
    df[DETAIL_DONE_COLUMN] = (
        df[DETAIL_DONE_COLUMN]
        .map(lambda x: bool(x) if not pd.isna(x) else False)
        .astype(bool)
    )

    records = df.to_dict(orient="records")
    _enrich_with_details(
        records,
        workers=workers,
        request_timeout=request_timeout,
        checkpoint_path=csv_path,
        checkpoint_every=checkpoint_every,
        log_every=log_every,
    )
    return pd.DataFrame(records)

In [ ]:
def _parse_cards_from_soup(soup: BeautifulSoup, official_newbuild_ids: Set[str]) -> List[dict]:
    cards = soup.find_all("div", class_="a-card")
    records = []
    for card in cards:
        if not card.get("data-id"):
            continue
        listing_id = str(card.get("data-id"))
        title_tag = card.find("a", class_="a-card__title")
        price_tag = card.find("div", class_="a-card__price")
        subtitle_tag = card.find("div", class_="a-card__subtitle")
        desc_tag = card.find("div", class_="a-card__text-preview")
        if not title_tag or not price_tag:
            continue
        raw_title = title_tag.get_text(strip=True)
        raw_price = price_tag.get_text(strip=True)
        raw_subtitle = subtitle_tag.get_text(strip=True) if subtitle_tag else ""
        raw_desc = desc_tag.get_text(strip=True) if desc_tag else ""

        parsed_title = _parse_title(raw_title)
        parsed_loc = _parse_location(raw_subtitle)
        desc_snippet = raw_desc[:300] if raw_desc else None
        snippet_facts = _parse_snippet_details(raw_desc)
        is_new = _classify_newbuild(listing_id, official_newbuild_ids)

        record = {
            "Listing_ID": listing_id,
            "Is_NewBuild": is_new,
            "Title": parsed_title["Title"],
            "Price_KZT": _parse_price(raw_price),
            "District": parsed_loc["District"],
            "Address": parsed_loc["Address"],
            "Floor_Info": parsed_title["Floor_Info"],
            "Area_m2": parsed_title["Area_m2"],
            "Rooms": parsed_title["Rooms"],
            "Build_Year": snippet_facts["Build_Year"],
            "Residential_Complex": snippet_facts["Residential_Complex"],
            "House_Type": snippet_facts["House_Type"],
            "Ceiling_Height": snippet_facts["Ceiling_Height"],
            "Bathroom": snippet_facts["Bathroom"],
            "Developer": None,
            "Description_Snippet": desc_snippet,
            "Listing_URL": "https://krisha.kz" + title_tag.get("href", ""),
            DETAIL_DONE_COLUMN: False,
        }
        records.append(record)
    return records


def scrape(
    pages: Optional[int] = None,
    output_path: str = "krisha_raw.csv",
    *,
    max_pages_cap: int = DEFAULT_MAX_PAGES_CAP,
    year_span: int = DEFAULT_BUILD_YEAR_SPAN,
    year_end: Optional[int] = None,
    fetch_details: bool = True,
    detail_workers: int = DEFAULT_DETAIL_WORKERS,
    max_listings: Optional[int] = None,
) -> pd.DataFrame:
    """Two-pass scrape with Krisha **construction-year** filter.

    Pass A — das[novostroiki]=1 + year window: collect official new-build IDs.
    Pass B — full segment (year-filtered); Is_NewBuild = membership in Pass A ID set only.
    Pass C — (optional, default ON) per-listing detail pages → Developer + missing facts.
    """
    year_params = _year_window_params(year_span, year_end)
    y_from = int(year_params["das[house.year][from]"])
    y_to = int(year_params["das[house.year][to]"])
    logger.info("Krisha filter: year of construction %d–%d (%d-year window, year_end=%d)", y_from, y_to, year_span, y_to)

    if pages is not None:
        logger.info("Pagination: fixed %d page(s) per pass", pages)
        page_iter_a = range(1, pages + 1)
        page_iter_b = range(1, pages + 1)
        auto_stop = False
    else:
        logger.info("Pagination: ALL pages until empty (max %d per pass, ~1–2.5s between requests)", max_pages_cap)
        page_iter_a = range(1, max_pages_cap + 1)
        page_iter_b = range(1, max_pages_cap + 1)
        auto_stop = True

    session = requests.Session()
    official_ids: Set[str] = set()

    logger.info("Pass A: new-build IDs (novostroiki + year)...")
    for page in page_iter_a:
        soup = _fetch_soup(session, page, newbuildings_only=True, year_params=year_params)
        if soup is None:
            logger.warning("Pass A: stopping at page %d (request failed)", page)
            break
        page_ids = _listing_ids_from_soup(soup)
        if not page_ids:
            if auto_stop:
                logger.info("Pass A: finished at page %d (0 listings — end of results)", page)
                break
            logger.info("  Page %d → +0 IDs", page)
        else:
            official_ids.update(page_ids)
            logger.info("  Page %d → +%d IDs (total unique %d)", page, len(page_ids), len(official_ids))
        if auto_stop and page >= max_pages_cap:
            if page_ids:
                logger.warning("Pass A: hit max_pages_cap=%d but still got listings", max_pages_cap)
            break
        time.sleep(random.uniform(1.0, 2.5))

    if max_listings is not None and max_listings > 0:
        logger.info("Pass B will stop early once %d listings are collected (max_listings cap)", max_listings)

    logger.info("Pass B: full catalog (year-filtered only)...")
    all_records: List[dict] = []
    for page in page_iter_b:
        soup = _fetch_soup(session, page, newbuildings_only=False, year_params=year_params)
        if soup is None:
            logger.warning("Pass B: stopping at page %d (request failed)", page)
            break
        recs = _parse_cards_from_soup(soup, official_ids)
        if not recs:
            if auto_stop:
                logger.info("Pass B: finished at page %d (0 listings — end of results)", page)
                break
            logger.info("Page %d → 0 listings", page)
        else:
            all_records.extend(recs)
            logger.info("Page %d → %d listings (cumulative %d)", page, len(recs), len(all_records))
        if max_listings is not None and len(all_records) >= max_listings:
            all_records = all_records[:max_listings]
            logger.info("Pass B: hit max_listings cap %d at page %d — stopping early", max_listings, page)
            break
        if auto_stop and page >= max_pages_cap:
            if recs:
                logger.warning("Pass B: hit max_pages_cap=%d but still got listings", max_pages_cap)
            break
        delay = random.uniform(1.0, 2.5)
        logger.info("Sleeping %.1fs before next page...", delay)
        time.sleep(delay)

    df = pd.DataFrame(all_records)
    if not df.empty:
        nb = int(df["Is_NewBuild"].sum())
        logger.info("Labels: %d new-build (%.1f%%), %d secondary", nb, 100 * nb / len(df), len(df) - nb)

    df.to_csv(output_path, index=False, encoding="utf-8-sig")
    logger.info("Saved %d raw rows (pre-details) to '%s'", len(df), output_path)

    if fetch_details and all_records:
        _enrich_with_details(
            all_records,
            workers=detail_workers,
            checkpoint_path=output_path,
        )
        df = pd.DataFrame(all_records)

    if not df.empty:
        for col in ("Build_Year", "Developer", "Residential_Complex", "House_Type", "Ceiling_Height", "Bathroom"):
            if col in df.columns:
                filled = int(df[col].notna().sum())
                logger.info("  %s populated: %d/%d (%.1f%%)", col, filled, len(df), 100 * filled / len(df))

    return df

### Quick test / resume Pass C (Task 1)

- Smoke-test на 2 страницах — раскомментируйте `scrape(pages=2, ...)`.
- **Resume** прерванного Pass C (детальные страницы) — раскомментируйте `enrich_csv_with_details(...)`. Будут досканированы только листинги с `Detail_Fetched=False`. Чекпоинт пишется в тот же `krisha_raw.csv` каждые 100 завершённых.
- Чтобы пропустить Pass C совсем (быстрее всего, без `Developer`), вызывайте `scrape(..., fetch_details=False)`.

Полный crawl запускается из секции **Run full pipeline**.

In [ ]:
# Smoke test (2 pages, year_span=2, no detail fetch — fast):
# df_raw_sample = scrape(pages=2, output_path="krisha_raw.csv", year_span=2, fetch_details=False)
# print(df_raw_sample.head())
# print(f"\nTotal rows scraped: {len(df_raw_sample)}")
# print(df_raw_sample["Is_NewBuild"].value_counts())

# Resume an interrupted Pass C on existing krisha_raw.csv:
# df_resumed = enrich_csv_with_details("krisha_raw.csv", workers=8)
# print(df_resumed[[\"Listing_ID\", \"Developer\", DETAIL_DONE_COLUMN]].head())

## 2. Task 2 — Preprocessing

Очистка `krisha_raw.csv` → `krisha_cleaned.csv`:
- разбираем `Floor_Info` на `Current_Floor` / `Total_Floors`;
- приводим типы (`Build_Year`, `Ceiling_Height`, `House_Type` и т.д.);
- удаляем дубликаты (по `Listing_URL` и по `Price+Area+Address`);
- срезаем outliers (площадь < 10 m², цена > 1 млрд KZT, IQR 1–99% по `Price_KZT`);
- считаем `Price_per_m2`;
- нормализуем названия районов до канонических 8 районов Алматы.

In [ ]:
def _coerce_bool_newbuild(series: pd.Series) -> pd.Series:
    """CSV-safe boolean coercion (string 'False' must NOT become True)."""
    def to_bool(x) -> bool:
        if pd.isna(x):
            return False
        if isinstance(x, (bool, np.bool_)):
            return bool(x)
        s = str(x).strip().lower()
        return s in ("true", "1", "yes", "1.0")
    return series.map(to_bool)


def _parse_floor_info(floor_str) -> Tuple[Optional[int], Optional[int]]:
    if pd.isna(floor_str) or not str(floor_str).strip():
        return None, None
    s = str(floor_str).strip()
    m = re.match(r"(\d+)\s+из\s+(\d+)", s)
    if m:
        return int(m.group(1)), int(m.group(2))
    m = re.match(r"(\d+)/(\d+)", s)
    if m:
        return int(m.group(1)), int(m.group(2))
    m = re.match(r"(\d+)", s)
    if m:
        return int(m.group(1)), None
    return None, None

In [ ]:
def clean(input_path: str = "krisha_raw.csv", output_path: str = "krisha_cleaned.csv") -> pd.DataFrame:
    df = pd.read_csv(input_path, encoding="utf-8-sig")
    initial_count = len(df)
    logger.info("Loaded %d rows from '%s'", initial_count, input_path)

    legacy_filter_cols = [c for c in ("Filter_Year_From", "Filter_Year_To") if c in df.columns]
    if legacy_filter_cols:
        df = df.drop(columns=legacy_filter_cols)
        logger.info("Dropped legacy filter columns: %s", ", ".join(legacy_filter_cols))

    if "Is_NewBuild" not in df.columns:
        df["Is_NewBuild"] = False
    else:
        df["Is_NewBuild"] = _coerce_bool_newbuild(df["Is_NewBuild"])

    if len(df) > 0 and df["Is_NewBuild"].sum() == 0 and "Listing_ID" not in df.columns:
        logger.warning(
            "Legacy CSV: no Listing_ID / all Is_NewBuild False — inferring новостройка from text. "
            "Re-scrape for labels from Krisha «новостройки» (official IDs)."
        )
        df["Is_NewBuild"] = df.apply(
            lambda r: infer_newbuild_from_text(
                str(r.get("Description_Snippet", "") or ""),
                str(r.get("Title", "") or ""),
            ),
            axis=1,
        )
        logger.info(
            "Heuristic labels: %d новостройка, %d вторичка",
            int(df["Is_NewBuild"].sum()),
            int(len(df) - int(df["Is_NewBuild"].sum())),
        )
    elif len(df) > 0 and df["Is_NewBuild"].sum() == 0 and "Listing_ID" in df.columns:
        logger.info("Is_NewBuild all False — keeping scraper labels (вторичка в разрезе Krisha).")

    floors = df["Floor_Info"].apply(_parse_floor_info)
    df["Current_Floor"] = floors.apply(lambda x: x[0]).astype("Int64")
    df["Total_Floors"] = floors.apply(lambda x: x[1]).astype("Int64")

    df["Price_KZT"] = pd.to_numeric(df["Price_KZT"], errors="coerce")
    df["Area_m2"] = pd.to_numeric(df["Area_m2"], errors="coerce")
    df["Rooms"] = pd.to_numeric(df["Rooms"], errors="coerce").astype("Int64")

    if "Build_Year" in df.columns:
        years = pd.to_numeric(df["Build_Year"], errors="coerce")
        current_year = pd.Timestamp.today().year
        years = years.where((years >= 1900) & (years <= current_year + 5))
        df["Build_Year"] = years.astype("Int64")
    if "Ceiling_Height" in df.columns:
        heights = pd.to_numeric(df["Ceiling_Height"], errors="coerce")
        df["Ceiling_Height"] = heights.where((heights >= 1.5) & (heights <= 6.0))
    for text_col in ("Residential_Complex", "House_Type", "Bathroom", "Developer"):
        if text_col in df.columns:
            df[text_col] = (
                df[text_col]
                .astype("string")
                .str.strip()
                .replace({"": pd.NA, "nan": pd.NA, "None": pd.NA})
            )

    df.dropna(subset=["Price_KZT", "Area_m2"], inplace=True)
    logger.info("After dropping missing Price/Area: %d rows", len(df))

    df.drop_duplicates(subset=["Listing_URL"], keep="first", inplace=True)
    df.drop_duplicates(subset=["Price_KZT", "Area_m2", "Address"], keep="first", inplace=True)
    logger.info("After deduplication: %d rows", len(df))

    df = df[df["Area_m2"] >= 10]
    df = df[df["Price_KZT"] <= 1_000_000_000]
    df = df[df["Price_KZT"] > 0]
    logger.info("After hard outlier removal: %d rows", len(df))

    Q1 = df["Price_KZT"].quantile(0.01)
    Q3 = df["Price_KZT"].quantile(0.99)
    df = df[(df["Price_KZT"] >= Q1) & (df["Price_KZT"] <= Q3)]
    logger.info("After IQR outlier removal (1-99 pct) on price: %d rows", len(df))

    df["Price_per_m2"] = (df["Price_KZT"] / df["Area_m2"]).round(0)

    KNOWN_DISTRICTS = {
        "Алатауский": "Алатауский р-н",
        "Алмалинский": "Алмалинский р-н",
        "Ауэзовский": "Ауэзовский р-н",
        "Бостандыкский": "Бостандыкский р-н",
        "Жетысуский": "Жетысуский р-н",
        "Медеуский": "Медеуский р-н",
        "Наурызбайский": "Наурызбайский р-н",
        "Турксибский": "Турксибский р-н",
    }

    def _normalize_district(d):
        if pd.isna(d):
            return "Прочее"
        for key, canonical in KNOWN_DISTRICTS.items():
            if key in str(d):
                return canonical
        return "Прочее"

    df["District"] = df["District"].apply(_normalize_district)

    df["Rooms"] = df["Rooms"].fillna(df["Rooms"].median())
    df["Address"] = df["Address"].fillna("Неизвестный")
    df["Description_Snippet"] = df["Description_Snippet"].fillna("")

    # Floor columns: leave as nullable Int64 (NaN acceptable for analysis)


    # ── 9b. Analysis-ready categoricals (grouping + light encoding) ───────────
    df["Market_Segment"] = np.where(df["Is_NewBuild"], "Новостройка", "Вторичка")

    try:
        df["Price_Tier"] = pd.qcut(
            df["Price_per_m2"],
            q=3,
            labels=["Недорогой сегмент", "Средний сегмент", "Премиум сегмент"],
            duplicates="drop",
        )
    except (ValueError, TypeError):
        df["Price_Tier"] = pd.Series(pd.NA, index=df.index, dtype="object")

    MIN_HOUSE_N = max(8, int(len(df) * 0.01))
    if "House_Type" in df.columns:
        ht = df["House_Type"].fillna("Не указано").astype(str).str.strip()
        vc_ht = ht.value_counts()
        rare_ht = set(vc_ht[vc_ht < MIN_HOUSE_N].index)
        df["House_Type_Group"] = ht.where(~ht.isin(rare_ht), "Прочее / редкий тип")
    else:
        df["House_Type_Group"] = "Не указано"

    MIN_BATH_N = max(10, int(len(df) * 0.015))
    if "Bathroom" in df.columns:
        bg = df["Bathroom"].fillna("Не указано").astype(str).str.strip()
        vc_b = bg.value_counts()
        rare_b = set(vc_b[vc_b < MIN_BATH_N].index) - {"Не указано"}
        df["Bathroom_Group"] = bg.where(~bg.isin(rare_b), "Прочее")
    else:
        df["Bathroom_Group"] = "Не указано"

    def _year_bucket(y) -> str:
        if pd.isna(y):
            return "Не указан"
        yi = int(y)
        if yi < 2000:
            return "до 2000"
        if yi < 2010:
            return "2000–2009"
        if yi < 2016:
            return "2010–2015"
        if yi < 2020:
            return "2016–2019"
        if yi < 2024:
            return "2020–2023"
        return "2024 и новее"

    if "Build_Year" in df.columns:
        df["Build_Year_Bucket"] = df["Build_Year"].apply(_year_bucket)
    else:
        df["Build_Year_Bucket"] = "Не указан"

    try:
        df["Area_Band"] = pd.qcut(
            df["Area_m2"],
            q=4,
            labels=["Маленькая", "Ниже средней", "Выше средней", "Большая"],
            duplicates="drop",
        )
    except (ValueError, TypeError):
        df["Area_Band"] = pd.Series(pd.NA, index=df.index, dtype="object")

    df["District_Code"] = pd.Categorical(df["District"]).codes
    df["Is_NewBuild_Code"] = df["Is_NewBuild"].astype(int)



    df.reset_index(drop=True, inplace=True)
    df.to_csv(output_path, index=False, encoding="utf-8-sig")

    removed = initial_count - len(df)
    logger.info(
        "Cleaned dataset: %d rows (removed %d / %.1f%%)",
        len(df), removed, removed / max(initial_count, 1) * 100,
    )
    logger.info("Saved to '%s'", output_path)
    return df

## 3. Task 3 — Analysis (5 questions)

1. **Q1** — Avg / median price per m² по районам.
2. **Q2** — Корреляция числа комнат с `Price_per_m2` (Pearson + Spearman).
3. **Q3** — Какой этаж дороже: первый, последний, средний.
4. **Q4** — Хрущёвки (≤ 5 этажей) vs Modern High-rise (≥ 12 этажей).
5. **Q5** — Топ-10 самых дорогих микрорайонов / улиц по `Price_per_m2`.

In [ ]:
def _classify_floor_level(row) -> Optional[str]:
    cur = row.get("Current_Floor")
    total = row.get("Total_Floors")
    if pd.isna(cur):
        return None
    if cur == 1:
        return "First"
    if not pd.isna(total) and cur == total:
        return "Last"
    return "Middle"


def _classify_building_type(total_floors) -> Optional[str]:
    if pd.isna(total_floors):
        return None
    if total_floors <= 5:
        return "Khrushchevka (≤5 floors)"
    if total_floors >= 12:
        return "Modern High-rise (12+ floors)"
    return "Mid-rise (6-11 floors)"


def analyze(input_path: str = "krisha_cleaned.csv") -> dict:
    df = pd.read_csv(input_path, encoding="utf-8-sig")
    logger.info("Loaded %d rows for analysis", len(df))
    results = {}

    q1 = (
        df.groupby("District")["Price_per_m2"]
        .agg(["mean", "median", "count"])
        .rename(columns={"mean": "Avg_Price_per_m2", "median": "Median_Price_per_m2", "count": "Listings"})
        .sort_values("Avg_Price_per_m2", ascending=False)
        .round(0)
    )
    results["q1"] = q1
    logger.info("Q1 — Districts ranked by avg price/m2:\n%s", q1.to_string())

    valid = df[["Rooms", "Price_per_m2"]].dropna()
    corr_pearson = valid["Rooms"].corr(valid["Price_per_m2"])
    corr_spearman = valid["Rooms"].corr(valid["Price_per_m2"], method="spearman")
    room_groups = (
        valid.groupby("Rooms")["Price_per_m2"]
        .agg(["mean", "count"])
        .rename(columns={"mean": "Avg_Price_per_m2", "count": "Listings"})
        .round(0)
    )
    results["q2"] = {
        "pearson_corr": round(corr_pearson, 4),
        "spearman_corr": round(corr_spearman, 4),
        "by_room_count": room_groups,
    }
    logger.info("Q2 — Rooms vs Price/m2: Pearson=%.4f, Spearman=%.4f", corr_pearson, corr_spearman)

    df["Floor_Level"] = df.apply(_classify_floor_level, axis=1)
    q3 = (
        df.dropna(subset=["Floor_Level"])
        .groupby("Floor_Level")["Price_KZT"]
        .agg(["mean", "median", "count"])
        .rename(columns={"mean": "Avg_Price_KZT", "median": "Median_Price_KZT", "count": "Listings"})
        .sort_values("Avg_Price_KZT", ascending=False)
        .round(0)
    )
    results["q3"] = q3
    logger.info("Q3 — Floor level prices:\n%s", q3.to_string())

    df["Building_Type"] = df["Total_Floors"].apply(_classify_building_type)
    q4 = (
        df.dropna(subset=["Building_Type"])
        .groupby("Building_Type")[["Price_KZT", "Price_per_m2"]]
        .agg(["mean", "median", "count"])
        .round(0)
    )
    q4.columns = ["_".join(c) for c in q4.columns]
    results["q4"] = q4
    logger.info("Q4 — Building types:\n%s", q4.to_string())

    df["Microdistrict"] = df["Address"].str.split(",").str[0].str.strip()
    q5 = (
        df.groupby("Microdistrict")["Price_per_m2"]
        .agg(["mean", "count"])
        .rename(columns={"mean": "Avg_Price_per_m2", "count": "Listings"})
        .query("Listings >= 3")
        .sort_values("Avg_Price_per_m2", ascending=False)
        .head(10)
        .round(0)
    )
    results["q5"] = q5
    logger.info("Q5 — Top 10 micro-districts by price/m2:\n%s", q5.to_string())
    return results

## 4. Task 4 — Visualization (10 charts)

**Charts 1–5** — общий обзор рынка (бокс-плот по районам, area vs price, heatmap, средняя ₸/m² по районам, гистограмма ₸/m²).

**Charts 6–10** — Новостройка vs Вторичка (mean price, ₸/m² boxplot, area vs price, stacked counts, overlay-гистограмма).

Все графики сохраняются в `charts/` (PNG + 2× HTML).

In [ ]:
matplotlib.use("Agg")

PALETTE = "husl"
FIGURE_DPI = 150
OUTPUT_DIR = Path("charts")
SEGMENT_NEW = "Новостройка"
SEGMENT_SEC = "Вторичка"


def _setup_output_dir():
    OUTPUT_DIR.mkdir(exist_ok=True)


def _with_segment_column(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    if "Is_NewBuild" not in out.columns:
        out["Is_NewBuild"] = False
    out["Is_NewBuild"] = out["Is_NewBuild"].fillna(False).astype(bool)
    out["Segment"] = np.where(out["Is_NewBuild"], SEGMENT_NEW, SEGMENT_SEC)
    return out


def _fmt_millions(x, _):
    if x >= 1_000_000:
        return f"{x/1_000_000:.0f}M"
    if x >= 1_000:
        return f"{x/1_000:.0f}K"
    return str(int(x))


def _fmt_price_per_m2_axis(x, _pos=None):
    xa = float(x)
    if abs(xa) >= 1_000_000:
        return f"{xa/1_000_000:.1f}M"
    if abs(xa) >= 1_000:
        return f"{xa/1_000:.0f}K"
    return f"{xa:.0f}"


def _newbuild_counts_line(df: pd.DataFrame) -> str:
    d = _with_segment_column(df)
    n_new = int(d["Is_NewBuild"].sum())
    n_sec = int(len(d) - n_new)
    return f"Новостройка: n={n_new}  |  Вторичка: n={n_sec}"

In [ ]:
def chart_boxplot_price_by_district(df: pd.DataFrame):
    """Chart 1 — Box plot of Price by District."""
    order = (
        df.groupby("District")["Price_KZT"].median().sort_values(ascending=False).index.tolist()
    )
    fig, ax = plt.subplots(figsize=(14, 7))
    sns.boxplot(data=df, x="District", y="Price_KZT", order=order, palette=PALETTE, showfliers=False, ax=ax)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(_fmt_millions))
    ax.set_title("Price Distribution by Almaty District", fontsize=16, fontweight="bold", pad=15)
    ax.set_xlabel("District", fontsize=12)
    ax.set_ylabel("Price (KZT)", fontsize=12)
    plt.xticks(rotation=30, ha="right", fontsize=9)
    plt.tight_layout()
    path = OUTPUT_DIR / "1_boxplot_price_by_district.png"
    fig.savefig(path, dpi=FIGURE_DPI)
    plt.close(fig)
    logger.info("Saved %s", path)


def chart_scatter_area_vs_price(df: pd.DataFrame):
    """Chart 2 — Area vs Price (colored by Rooms), HTML + PNG."""
    plot_df = df.dropna(subset=["Area_m2", "Price_KZT", "Rooms"]).copy()
    plot_df["Rooms_str"] = plot_df["Rooms"].astype(int).astype(str) + "-комн."
    fig = px.scatter(
        plot_df,
        x="Area_m2",
        y="Price_KZT",
        color="Rooms_str",
        hover_data=["District", "Address", "Floor_Info"],
        title="Area vs Price — Colored by Room Count",
        labels={"Area_m2": "Area (m²)", "Price_KZT": "Price (KZT)", "Rooms_str": "Rooms"},
        template="plotly_white",
        opacity=0.65,
    )
    fig.update_layout(title_font_size=18, xaxis_title_font_size=13, yaxis_title_font_size=13, legend_title_font_size=12)
    path = OUTPUT_DIR / "2_scatter_area_vs_price.html"
    fig.write_html(str(path))
    logger.info("Saved %s", path)

    static_fig, ax = plt.subplots(figsize=(12, 7))
    rooms_sorted = sorted(plot_df["Rooms"].unique())
    colors = sns.color_palette(PALETTE, len(rooms_sorted))
    for room, color in zip(rooms_sorted, colors):
        subset = plot_df[plot_df["Rooms"] == room]
        ax.scatter(subset["Area_m2"], subset["Price_KZT"], label=f"{int(room)}-комн.", alpha=0.55, s=25, color=color)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(_fmt_millions))
    ax.set_title("Area vs Price (colored by Room Count)", fontsize=15, fontweight="bold")
    ax.set_xlabel("Area (m²)", fontsize=12)
    ax.set_ylabel("Price (KZT)", fontsize=12)
    ax.legend(title="Rooms", bbox_to_anchor=(1.01, 1), loc="upper left")
    plt.tight_layout()
    png_path = OUTPUT_DIR / "2_scatter_area_vs_price.png"
    static_fig.savefig(png_path, dpi=FIGURE_DPI)
    plt.close(static_fig)
    logger.info("Saved %s", png_path)


def chart_heatmap_correlation(df: pd.DataFrame):
    """Chart 3 — Correlation heatmap."""
    cols = ["Price_KZT", "Area_m2", "Rooms", "Current_Floor", "Total_Floors", "Price_per_m2"]
    corr_df = df[cols].dropna().corr().round(2)
    fig, ax = plt.subplots(figsize=(9, 7))
    sns.heatmap(corr_df, annot=True, fmt=".2f", cmap="RdYlGn", vmin=-1, vmax=1, linewidths=0.5, ax=ax, annot_kws={"size": 11})
    ax.set_title("Correlation Heatmap: Price, Area, Rooms, Floor", fontsize=14, fontweight="bold", pad=12)
    plt.tight_layout()
    path = OUTPUT_DIR / "3_heatmap_correlation.png"
    fig.savefig(path, dpi=FIGURE_DPI)
    plt.close(fig)
    logger.info("Saved %s", path)


def chart_bar_avg_price_per_m2_by_district(df: pd.DataFrame):
    """Chart 4 — Bar chart of avg ₸/m² by district."""
    agg = (
        df.groupby("District")["Price_per_m2"].mean().sort_values(ascending=True).reset_index()
    )
    fig, ax = plt.subplots(figsize=(11, 6))
    bars = ax.barh(agg["District"], agg["Price_per_m2"], color=sns.color_palette(PALETTE, len(agg)), edgecolor="white", linewidth=0.5)
    for bar, val in zip(bars, agg["Price_per_m2"]):
        ax.text(bar.get_width() + 50, bar.get_y() + bar.get_height() / 2, f"{val:,.0f} ₸/m²", va="center", fontsize=9)
    ax.set_title("Average Price per m² by District (Sorted)", fontsize=15, fontweight="bold")
    ax.set_xlabel("Price per m² (KZT)", fontsize=12)
    ax.set_ylabel("")
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(_fmt_millions))
    plt.tight_layout()
    path = OUTPUT_DIR / "4_bar_avg_price_per_m2_district.png"
    fig.savefig(path, dpi=FIGURE_DPI)
    plt.close(fig)
    logger.info("Saved %s", path)


def chart_histogram_price_per_m2(df: pd.DataFrame):
    """Chart 5 — Histogram of Price_per_m2."""
    data = df["Price_per_m2"].dropna()
    fig, ax = plt.subplots(figsize=(12, 6))
    n, bins, patches = ax.hist(data, bins=60, color="#4C9BE8", edgecolor="white", linewidth=0.4, alpha=0.85)
    p25, p75 = data.quantile(0.25), data.quantile(0.75)
    for patch, left in zip(patches, bins[:-1]):
        if p25 <= left <= p75:
            patch.set_facecolor("#E8844C")
            patch.set_alpha(0.9)
    ax.axvline(data.median(), color="#d62728", linestyle="--", linewidth=1.8, label=f"Median: {data.median():,.0f} ₸/m²")
    ax.axvline(data.mean(), color="#2ca02c", linestyle="--", linewidth=1.8, label=f"Mean: {data.mean():,.0f} ₸/m²")
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(_fmt_millions))
    ax.set_title("Distribution of Price per m² — Market Sweet Spots (orange = IQR)", fontsize=14, fontweight="bold")
    ax.set_xlabel("Price per m² (KZT)", fontsize=12)
    ax.set_ylabel("Number of Listings", fontsize=12)
    ax.legend(fontsize=11)
    plt.tight_layout()
    path = OUTPUT_DIR / "5_histogram_price_per_m2.png"
    fig.savefig(path, dpi=FIGURE_DPI)
    plt.close(fig)
    logger.info("Saved %s", path)

In [ ]:
def chart_nb_bar_mean_price_and_m2(df: pd.DataFrame):
    """Chart 6 — Mean total price + mean ₸/m² by segment."""
    d = _with_segment_column(df)
    order = [SEGMENT_NEW, SEGMENT_SEC]
    agg = d.groupby("Segment")[["Price_KZT", "Price_per_m2"]].mean().reindex(order).fillna(0)
    counts = d.groupby("Segment").size().reindex(order).fillna(0).astype(int)

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    colors_p = ["#2ecc71", "#e74c3c"]
    colors_m = ["#3498db", "#9b59b6"]

    x = np.arange(len(order))
    axes[0].bar(x, agg["Price_KZT"].values, color=colors_p, edgecolor="white")
    axes[0].set_xticks(x)
    axes[0].set_xticklabels([f"{s}\n(n={counts[s]})" for s in order], fontsize=10)
    axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(_fmt_millions))
    axes[0].set_title("Mean Price (KZT) by Segment", fontweight="bold")
    axes[0].set_ylabel("KZT")

    axes[1].bar(x, agg["Price_per_m2"].values, color=colors_m, edgecolor="white")
    axes[1].set_xticks(x)
    axes[1].set_xticklabels([f"{s}\n(n={counts[s]})" for s in order], fontsize=10)
    axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(_fmt_price_per_m2_axis))
    axes[1].set_title("Mean Price per m² by Segment", fontweight="bold")
    axes[1].set_ylabel("₸/m²")

    fig.suptitle(
        "New-Build vs Secondary — Average Price Comparison\n" + _newbuild_counts_line(df),
        fontsize=13, fontweight="bold", y=1.05,
    )
    plt.tight_layout()
    path = OUTPUT_DIR / "6_newbuild_bar_mean_price.png"
    fig.savefig(path, dpi=FIGURE_DPI, bbox_inches="tight")
    plt.close(fig)
    logger.info("Saved %s", path)


def chart_nb_boxplot_price_per_m2(df: pd.DataFrame):
    """Chart 7 — Box plot of ₸/m² by segment."""
    d = _with_segment_column(df)
    fig, ax = plt.subplots(figsize=(8, 6))
    sns.boxplot(
        data=d,
        x="Segment",
        y="Price_per_m2",
        order=[SEGMENT_NEW, SEGMENT_SEC],
        palette=["#2ecc71", "#e74c3c"],
        showfliers=False,
        ax=ax,
    )
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(_fmt_price_per_m2_axis))
    ax.set_title("Price per m²: New-Build vs Secondary\n" + _newbuild_counts_line(df), fontsize=13, fontweight="bold")
    ax.set_xlabel("")
    plt.tight_layout()
    path = OUTPUT_DIR / "7_newbuild_boxplot_price_per_m2.png"
    fig.savefig(path, dpi=FIGURE_DPI)
    plt.close(fig)
    logger.info("Saved %s", path)


def chart_nb_scatter_area_price(df: pd.DataFrame):
    """Chart 8 — Area vs Price by segment (HTML + PNG)."""
    d = _with_segment_column(df).dropna(subset=["Area_m2", "Price_KZT"])
    fig = px.scatter(
        d,
        x="Area_m2",
        y="Price_KZT",
        color="Segment",
        color_discrete_map={SEGMENT_NEW: "#2ecc71", SEGMENT_SEC: "#e74c3c"},
        hover_data=["District", "Price_per_m2", "Listing_URL"] if "Listing_URL" in d.columns else ["District", "Price_per_m2"],
        title="Area vs Price — New-Build vs Secondary<br><sup>" + _newbuild_counts_line(df) + "</sup>",
        labels={"Area_m2": "Area (m²)", "Price_KZT": "Price (KZT)"},
        template="plotly_white",
        opacity=0.65,
    )
    fig.write_html(str(OUTPUT_DIR / "8_newbuild_scatter_area_vs_price.html"))
    logger.info("Saved %s", OUTPUT_DIR / "8_newbuild_scatter_area_vs_price.html")

    fig2, ax = plt.subplots(figsize=(11, 7))
    for seg, color in [(SEGMENT_NEW, "#2ecc71"), (SEGMENT_SEC, "#e74c3c")]:
        sub = d[d["Segment"] == seg]
        ax.scatter(sub["Area_m2"], sub["Price_KZT"], label=seg, alpha=0.5, s=28, color=color, edgecolors="none")
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(_fmt_millions))
    ax.set_title("Area vs Price (New-Build vs Secondary)\n" + _newbuild_counts_line(df), fontsize=13, fontweight="bold")
    ax.set_xlabel("Area (m²)")
    ax.set_ylabel("Price (KZT)")
    ax.legend(title="Сегмент")
    plt.tight_layout()
    path = OUTPUT_DIR / "8_newbuild_scatter_area_vs_price.png"
    fig2.savefig(path, dpi=FIGURE_DPI)
    plt.close(fig2)
    logger.info("Saved %s", path)


def chart_nb_stacked_counts_by_district(df: pd.DataFrame):
    """Chart 9 — Stacked bar: listings per top district by segment."""
    d = _with_segment_column(df)
    top = d["District"].value_counts().head(8).index.tolist()
    sub = d[d["District"].isin(top)]
    ct = pd.crosstab(sub["District"], sub["Segment"])
    for col in [SEGMENT_NEW, SEGMENT_SEC]:
        if col not in ct.columns:
            ct[col] = 0
    ct = ct[[SEGMENT_NEW, SEGMENT_SEC]]

    fig, ax = plt.subplots(figsize=(12, 6))
    ct.plot(kind="bar", stacked=True, ax=ax, color=["#2ecc71", "#e74c3c"], edgecolor="white", linewidth=0.5)
    ax.set_title("Listings by District — New-Build vs Secondary (stacked)\n" + _newbuild_counts_line(df), fontsize=12, fontweight="bold")
    ax.set_xlabel("District")
    ax.set_ylabel("Number of listings")
    ax.legend(title="Сегмент")
    plt.xticks(rotation=25, ha="right")
    plt.tight_layout()
    path = OUTPUT_DIR / "9_newbuild_stacked_counts_by_district.png"
    fig.savefig(path, dpi=FIGURE_DPI)
    plt.close(fig)
    logger.info("Saved %s", path)


def chart_nb_hist_price_per_m2_overlay(df: pd.DataFrame):
    """Chart 10 — Overlay histogram of ₸/m² by segment (density)."""
    fig, ax = plt.subplots(figsize=(11, 6))
    nb = _with_segment_column(df)
    d_new = nb.loc[nb["Is_NewBuild"], "Price_per_m2"].dropna()
    d_sec = nb.loc[~nb["Is_NewBuild"], "Price_per_m2"].dropna()
    bins = 45
    ax.hist(d_sec, bins=bins, alpha=0.55, color="#e74c3c", label=SEGMENT_SEC, density=True)
    ax.hist(d_new, bins=bins, alpha=0.55, color="#2ecc71", label=SEGMENT_NEW, density=True)
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(_fmt_price_per_m2_axis))
    ax.set_title("Price per m² Distribution (normalized) — New-Build vs Secondary\n" + _newbuild_counts_line(df), fontsize=12, fontweight="bold")
    ax.set_xlabel("Price per m² (KZT)")
    ax.set_ylabel("Density")
    ax.legend()
    plt.tight_layout()
    path = OUTPUT_DIR / "10_newbuild_hist_price_per_m2_overlay.png"
    fig.savefig(path, dpi=FIGURE_DPI)
    plt.close(fig)
    logger.info("Saved %s", path)


def visualize(input_path: str = "krisha_cleaned.csv"):
    """Generate 5 general + 5 new-build vs secondary charts to charts/."""
    _setup_output_dir()
    df = pd.read_csv(input_path, encoding="utf-8-sig")
    logger.info("Loaded %d rows for visualization", len(df))

    chart_boxplot_price_by_district(df)
    chart_scatter_area_vs_price(df)
    chart_heatmap_correlation(df)
    chart_bar_avg_price_per_m2_by_district(df)
    chart_histogram_price_per_m2(df)

    chart_nb_bar_mean_price_and_m2(df)
    chart_nb_boxplot_price_per_m2(df)
    chart_nb_scatter_area_price(df)
    chart_nb_stacked_counts_by_district(df)
    chart_nb_hist_price_per_m2_overlay(df)

    logger.info("All charts saved to '%s/' (1–10)", OUTPUT_DIR)

## 5. Task 5 — Storytelling summary report

Печатает ключевые insights и turning points для слайдов / презентации.

In [ ]:
def print_summary_report(results: dict, df: pd.DataFrame):
    """Print key insights and 'turning points' for presentation slides."""
    sep = "=" * 70
    print(f"\n{sep}")
    print("  ALMATY HOUSING MARKET — KEY INSIGHTS & TURNING POINTS")
    print(sep)

    q1: pd.DataFrame = results["q1"]
    if len(q1) >= 2:
        most_exp = q1.index[0]
        cheapest = q1.index[-1]
        most_exp_val = q1.loc[most_exp, "Avg_Price_per_m2"]
        cheapest_val = q1.loc[cheapest, "Avg_Price_per_m2"]
        pct_diff = (most_exp_val - cheapest_val) / cheapest_val * 100
        print(f"\n[District Prices]")
        print(f"  • Most expensive district : {most_exp} — {most_exp_val:,.0f} ₸/m²")
        print(f"  • Most affordable district: {cheapest} — {cheapest_val:,.0f} ₸/m²")
        print(f"  ★ TURNING POINT: {most_exp} is {pct_diff:.0f}% MORE expensive than {cheapest}")

        central_keywords = ["Алмалинский", "Медеуский", "Бостандыкский"]
        for kw in central_keywords:
            match = [d for d in q1.index if kw in d]
            if match:
                d = match[0]
                rank = list(q1.index).index(d) + 1
                print(f"  ★ {d} ranks #{rank} out of {len(q1)} districts by price/m²")

    q2 = results["q2"]
    pearson = q2["pearson_corr"]
    direction = "NEGATIVE" if pearson < 0 else "POSITIVE"
    strength = "weak" if abs(pearson) < 0.3 else ("moderate" if abs(pearson) < 0.6 else "strong")
    print(f"\n[Rooms vs Price/m²]")
    print(f"  • Pearson correlation: {pearson:.3f} ({strength} {direction})")
    if pearson < -0.1:
        print("  ★ TURNING POINT: More rooms = CHEAPER per m² — smaller units command a premium!")
    elif pearson > 0.1:
        print("  ★ TURNING POINT: Larger apartments command higher price per m² — luxury effect.")
    else:
        print("  ★ Room count has little independent effect on price/m².")

    q3: pd.DataFrame = results["q3"]
    if not q3.empty:
        top_floor_level = q3.index[0]
        print(f"\n[Floor Level Analysis]")
        for level in q3.index:
            print(f"  • {level} floor: avg {q3.loc[level,'Avg_Price_KZT']:,.0f} ₸")
        print(f"  ★ TURNING POINT: {top_floor_level} floors are the most expensive on average.")

    q4: pd.DataFrame = results["q4"]
    if not q4.empty:
        print(f"\n[Building Type Comparison]")
        types = q4.index.tolist()
        khrush = [t for t in types if "Khrushchevka" in t]
        modern = [t for t in types if "High-rise" in t]
        if khrush and modern:
            k_price = q4.loc[khrush[0], "Price_per_m2_mean"]
            m_price = q4.loc[modern[0], "Price_per_m2_mean"]
            pct = (m_price - k_price) / k_price * 100
            print(f"  • Khrushchevka (≤5 fl.) : avg {k_price:,.0f} ₸/m²")
            print(f"  • Modern High-rise (12+) : avg {m_price:,.0f} ₸/m²")
            if pct > 0:
                print(f"  ★ TURNING POINT: Modern high-rises cost {pct:.0f}% MORE than Khrushchevkas per m².")
            else:
                print(
                    f"  ★ TURNING POINT: Khrushchevkas actually cost {abs(pct):.0f}% MORE per m² — "
                    f"likely due to central location premium!"
                )

    q5: pd.DataFrame = results["q5"]
    if not q5.empty:
        print(f"\n[Top 5 Most Expensive Micro-districts/Streets]")
        for i, (idx, row) in enumerate(q5.head(5).iterrows(), 1):
            print(f"  {i}. {idx} — {row['Avg_Price_per_m2']:,.0f} ₸/m²  ({int(row['Listings'])} listings)")
        print(f"  ★ The #1 most expensive location is '{q5.index[0]}'")

    print(f"\n[Market Overview]")
    print(f"  • Total listings analyzed : {len(df):,}")
    print(f"  • Median price            : {df['Price_KZT'].median():,.0f} ₸")
    print(f"  • Median price per m²     : {df['Price_per_m2'].median():,.0f} ₸/m²")
    print(f"  • Median area             : {df['Area_m2'].median():.1f} m²")
    print(f"  • Districts covered       : {df['District'].nunique()}")

    if "Is_NewBuild" in df.columns:
        nb = df["Is_NewBuild"].astype(bool)
        n_nb, n_sec = int(nb.sum()), int((~nb).sum())
        print(f"\n[New build vs secondary (labeled)]")
        print(f"  • Новостройка : {n_nb:,} listings ({100 * n_nb / len(df):.1f}%)")
        print(f"  • Вторичка    : {n_sec:,} listings ({100 * n_sec / len(df):.1f}%)")
        if n_nb > 0 and n_sec > 0:
            m_nb = df.loc[nb, "Price_per_m2"].median()
            m_sec = df.loc[~nb, "Price_per_m2"].median()
            if m_sec > 0:
                pct = (m_nb - m_sec) / m_sec * 100
                print(f"  • Median ₸/m²: новостройка {m_nb:,.0f} vs вторичка {m_sec:,.0f} ({pct:+.1f}% vs secondary)")
            print("  ★ TURNING POINT: Compare segments in charts 6–10 (new-build split).")

    print(f"\n{sep}")
    print("  Charts: charts/1–5 (market) + charts/6–10 (new vs secondary).")
    print("  Full pipeline log in 'pipeline.log'.")
    print(sep)

## 6. Run full pipeline

Запускает Scrape → Clean → Analyze → Visualize → Summary report.

- **Year window** = последние **2** календарных года (`BUILD_YEAR_SPAN = 2` в первой ячейке), т.е. сейчас 2025–2026.
- **Cap на размер выборки** = `SCRAPE_MAX_LISTINGS = 2000` (Pass B остановится сам, как соберёт 2000 листингов). Поставьте `None` для полного crawl.
- `krisha_raw.csv` сохраняется **сразу после Pass B**, ещё до детальных страниц. Ctrl+C во время Pass C ничего не уничтожит.
- Pass C (детальные страницы) **резюмируемый**: чекпоинт каждые 100 листингов; повторный запуск дочистит только `Detail_Fetched=False`.
- Если `krisha_raw.csv` уже есть, `run_pipeline()` пропустит scrape (удалите файл, чтобы перескрапить с нуля).

In [ ]:
def run_pipeline():
    logger.info("╔══ Starting Almaty Housing Market Analysis Pipeline ══╗")

    if Path(RAW_CSV).exists():
        logger.info("'%s' already exists — skipping scrape. Delete to re-scrape.", RAW_CSV)
    else:
        if SCRAPE_PAGES is None:
            logger.info(
                "Task 1: Full crawl — year window %d yrs, 2 passes, until empty (cap %s pages/pass)...",
                BUILD_YEAR_SPAN,
                f"{SCRAPE_MAX_PAGES_CAP:,}",
            )
        else:
            logger.info(
                "Task 1: Scraping krisha.kz (%d pages × 2 passes, year window %d yrs)...",
                SCRAPE_PAGES,
                BUILD_YEAR_SPAN,
            )
        scrape(
            pages=SCRAPE_PAGES,
            output_path=RAW_CSV,
            year_span=BUILD_YEAR_SPAN,
            max_pages_cap=SCRAPE_MAX_PAGES_CAP,
            max_listings=SCRAPE_MAX_LISTINGS,
        )

    raw_df = pd.read_csv(RAW_CSV, encoding="utf-8-sig")
    logger.info("Raw data: %d rows", len(raw_df))
    if len(raw_df) < 500:
        logger.warning(
            "Only %d rows in raw data — for a course sample you may want more pages or a wider year window.",
            len(raw_df),
        )

    logger.info("Task 2: Cleaning data...")
    df = clean(input_path=RAW_CSV, output_path=CLEAN_CSV)

    logger.info("Task 3: Running analysis...")
    results = analyze(input_path=CLEAN_CSV)

    logger.info("Task 4: Generating visualizations...")
    visualize(input_path=CLEAN_CSV)

    print_summary_report(results, df)
    logger.info("╚══ Pipeline complete ══╝")
    return df, results

In [ ]:
df, results = run_pipeline()